# 02 SPC and capability: crimp height

X-bar and R charts per machine with Western Electric rules on the simulated crimp height data
(`data/simulated/crimp_height.csv`, built by `python -m src.simulate.crimp`).

This notebook reads only the analysis table. The hidden truth file (`crimp_truth.json`) is not
opened here: the charts have to find the special causes on their own.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # make `src` importable when run from notebooks/

In [ ]:
import pandas as pd

from src.plots import xbar_r_chart
from src.spc import chart_table, signal_log

FIGURES = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(ROOT / "data" / "simulated" / "crimp_height.csv", dtype={"shift": str})
print(
    len(data),
    "parts;",
    data["subgroup"].nunique(),
    "subgroups of",
    data.groupby("subgroup").size().iloc[0],
)
data.head()

## 1. Phase I baseline and limits

Each machine gets its own charts. Limits come from the first 5 production days (30 subgroups
per machine), the Phase I baseline; all 180 subgroups per machine are then checked against
those limits (Phase II). Signals inside the baseline are kept in the log: in a real plant each
one would be investigated before the limits are frozen.

In [ ]:
BASELINE_DAYS = 5
tables, limits = {}, {}
for machine, df in data.groupby("machine"):
    stats_days = (
        df.drop_duplicates("subgroup").sort_values("subgroup")["day"].reset_index(drop=True)
    )
    table, lim = chart_table(df, baseline=stats_days <= BASELINE_DAYS)
    tables[machine], limits[machine] = table, lim

limits_table = (
    pd.DataFrame(limits)
    .T[["xbar_center", "xbar_lcl", "xbar_ucl", "r_center", "r_ucl", "sigma_hat"]]
    .astype(float)
    .round(4)
)
limits_table.to_csv(TABLES / "spc_limits.csv")
limits_table

## 2. Charts

In [ ]:
for machine, table in tables.items():
    xbar_r_chart(
        table,
        limits[machine],
        f"Machine {machine}: X-bar and R charts, crimp height",
        FIGURES / f"spc_xbar_r_machine_{machine}.png",
        eyebrow="SPC · Crimp height",
        source="Simulated data (seeded), not plant measurements",
    )

![Machine A](../reports/figures/spc_xbar_r_machine_A.png)

![Machine B](../reports/figures/spc_xbar_r_machine_B.png)

## 3. Signal log

In [ ]:
log = pd.concat([signal_log(t) for t in tables.values()], ignore_index=True)
log.to_csv(TABLES / "spc_signal_log.csv", index=False)
summary = log.groupby(["machine", "chart"]).agg(
    signals=("subgroup", "size"), first_subgroup=("subgroup", "min"), first_day=("day", "min")
)
summary

In [ ]:
# First signal on each chart after the baseline, and how long signals persist from there.
after = log[~log["baseline"]]
after.groupby(["machine", "chart"]).head(3)

## 4. Stratification by shift

The R chart pools all shifts, so a spread problem on one shift shows up as scattered range
signals. Grouping subgroup ranges by shift makes the pattern visible. sigma = R-bar / d2.

In [ ]:
from src.spc import d2

ranges = pd.concat(tables.values())
by_shift = (
    ranges.groupby(["machine", "shift"])["range"]
    .mean()
    .rename("r_bar")
    .to_frame()
    .assign(sigma_hat=lambda t: t["r_bar"] / d2(5))
)
by_shift["ratio_to_shift_1"] = (
    by_shift["sigma_hat"]
    / by_shift.xs("1", level="shift")["sigma_hat"]
    .reindex(by_shift.index.get_level_values("machine"))
    .to_numpy()
)
r_signals_by_shift = (
    log[log["chart"] == "R"].groupby(["machine", "shift"]).size().rename("r_signals")
)
by_shift = by_shift.join(r_signals_by_shift).fillna({"r_signals": 0}).astype({"r_signals": int})
by_shift.round(4).to_csv(TABLES / "spc_range_by_shift.csv")
by_shift.round(4)

## 5. Normality

Cp, Cpk, Pp and Ppk assume normally distributed data (NIST 6.1.6). Anderson-Darling test per
data set, plus a normal probability plot. The data are rounded to the gauge resolution
(0.001 mm), which leaves only a few dozen distinct values; formal tests on large, rounded
samples reject normality more easily, so the plot is read together with the test.

In [ ]:
from src.capability import anderson_darling_normal, capability_study
from src.plots import capability_histograms, normal_probability_plot

LSL, USL = 1.75, 1.85
VALUE = "crimp_height_mm"

## 6. Capability before and after removing special causes

Special causes are removed on the evidence of section 3 and 4 only:

- **Machine B from subgroup 182 on:** the X-bar chart signals on every machine B subgroup from
  there to the end, so the process mean moved at that point.
- **Shift 3 on both machines:** all R chart signals fall on shift 3, and its sigma estimate is
  well above shifts 1 and 2.

What is left is the process as it runs without those causes.

In [ ]:
special_mask = ((data["machine"] == "B") & (data["subgroup"] >= 182)) | (data["shift"] == "3")
clean = data[~special_mask]

sets = {
    "All data, before": data,
    "Machine A, before": data[data["machine"] == "A"],
    "Machine B, before": data[data["machine"] == "B"],
    "All data, after removal": clean,
    "Machine A, after removal": clean[clean["machine"] == "A"],
    "Machine B, after removal": clean[clean["machine"] == "B"],
}
rows = []
for name, df in sets.items():
    cap = capability_study(df, VALUE, "subgroup", LSL, USL)
    ad = anderson_darling_normal(df[VALUE])
    rows.append(
        {
            "data set": name,
            **cap,
            "ad_a2": ad["a2"],
            "ad_p": ad["p_value"],
            "distinct_values": df[VALUE].nunique(),
        }
    )
capability = pd.DataFrame(rows).set_index("data set")
capability.round(4).to_csv(TABLES / "capability.csv")
capability[
    [
        "n",
        "mean",
        "sigma_within",
        "sigma_overall",
        "cp",
        "cpk",
        "cpk_low_95",
        "cpk_high_95",
        "pp",
        "ppk",
        "ad_p",
        "distinct_values",
    ]
].round(3)

In [ ]:
capability_histograms(
    [
        ("All data, before removing special causes", data[VALUE]),
        ("After removing machine B from subgroup 182 and shift 3", clean[VALUE]),
    ],
    LSL,
    USL,
    "Crimp height against the specification",
    FIGURES / "capability_histograms.png",
    eyebrow="Capability · Crimp height",
    source="Simulated data (seeded), not plant measurements",
)
normal_probability_plot(
    clean.loc[clean["machine"] == "A", VALUE],
    "Normal probability plot, machine A after removal",
    FIGURES / "capability_normal_probability_A.png",
    eyebrow="Capability · Normality check",
    source="Simulated data (seeded), not plant measurements",
)

![Histograms](../reports/figures/capability_histograms.png)

![Normal probability plot](../reports/figures/capability_normal_probability_A.png)

## 7. Scoring against the simulated truth

Only this section opens `crimp_truth.json`. Each subgroup gets its true label, and the chart
signals from section 3 are scored against it.

- **Mean shift** (X-bar chart): special = machine B from the true first shifted subgroup.
  In-control subgroups for the false alarm rate exclude both causes.
- **Variance increase** (R chart): special = shift 3.
- NIST (6.3.2) gives an average of one false alarm in about 91.75 points with the Western
  Electric rules, against one in 371 for the 3-sigma rule alone.

In [ ]:
import json

from src.capability import detection_delay, score_signals

truth = json.loads((ROOT / "data" / "simulated" / "crimp_truth.json").read_text())
shift_cause, var_cause = truth["special_causes"]

charts = pd.concat(tables.values(), ignore_index=True)
charts["mean_shift"] = (charts["machine"] == shift_cause["machine"]) & (
    charts["subgroup"] >= shift_cause["first_subgroup"]
)
charts["high_variance"] = charts["shift"] == var_cause["shift"]
charts["x_signal"] = charts["xbar_rules"] != ""
charts["r_signal"] = charts["r_rules"] != ""

in_control = ~charts["mean_shift"] & ~charts["high_variance"]
b = charts[charts["machine"] == shift_cause["machine"]]
scores = pd.DataFrame(
    {
        "X-bar vs mean shift (machine B)": score_signals(b["x_signal"], b["mean_shift"]),
        "X-bar false alarms (in-control subgroups)": score_signals(
            charts.loc[in_control, "x_signal"], charts.loc[in_control, "mean_shift"]
        ),
        "R vs shift 3 variance (both machines)": score_signals(
            charts["r_signal"], charts["high_variance"]
        ),
    }
).T
delay = detection_delay(b["subgroup"], b["x_signal"], shift_cause["first_subgroup"])
print("Machine B subgroups between the tool change and the first X-bar signal:", delay)
scores.to_csv(TABLES / "detection_scores.csv")
scores

In [ ]:
# Capability estimated from data vs the true capability of each condition.
true_vs_est = []
cond = {
    "Base process (machine A, shifts 1-2)": (data["machine"] == "A") & (data["shift"] != "3"),
    "Machine B after tool change (shifts 1-2)": (data["machine"] == "B")
    & (data["subgroup"] >= shift_cause["first_subgroup"])
    & (data["shift"] != "3"),
    "Shift 3 (machine A)": (data["machine"] == "A") & (data["shift"] == "3"),
}
true_cpk = {
    "Base process (machine A, shifts 1-2)": truth["base_process"]["cpk"],
    "Machine B after tool change (shifts 1-2)": shift_cause["cpk_after"],
    "Shift 3 (machine A)": var_cause["cpk_on_shift"],
}
for name, mask in cond.items():
    cap = capability_study(data[mask], VALUE, "subgroup", LSL, USL)
    true_vs_est.append(
        {
            "condition": name,
            "true_cpk": true_cpk[name],
            "cpk": cap["cpk"],
            "cpk_low_95": cap["cpk_low_95"],
            "cpk_high_95": cap["cpk_high_95"],
            "n": cap["n"],
        }
    )
true_vs_est = pd.DataFrame(true_vs_est).set_index("condition")
true_vs_est["truth_inside_95"] = true_vs_est["true_cpk"].between(
    true_vs_est["cpk_low_95"], true_vs_est["cpk_high_95"]
)
true_vs_est.round(3).to_csv(TABLES / "capability_vs_truth.csv")
true_vs_est.round(3)